# PBPK Enriched Dataset — Exploratory Analysis

Este notebook documenta estatísticas e correlações principais do dataset enriquecido `pbpk_parameters_wide_enriched_v3.csv`, utilizado para calibrar e avaliar o DynamicPBPKGNN.



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')
BASE_DIR = Path("../")
DATA_PATH = BASE_DIR / "analysis" / "pbpk_parameters_wide_enriched_v3.csv"



In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Amostras: {len(df):,}")

def resolve_clearance(row):
    for col in ["clearance_hepatic_l_h", "clearance_l_h", "microsome_hepatic_l_h"]:
        val = row.get(col)
        if pd.notna(val):
            return float(val)
    return np.nan


df["clearance_target"] = df.apply(resolve_clearance, axis=1)
df["fu_frac"] = df["fu_frac"].clip(lower=0.0, upper=1.0)
df.head()


In [ ]:
summary_cols = ["clearance_target", "microsome_clint_l_h", "fu_frac", "vd_l_kg", "bioavailability_frac"]
summary = df[summary_cols].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).T
summary


In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["clearance_target"].dropna(), bins=50, kde=True, color="#1f77b4")
plt.xlabel("Clearance (L/h)")
plt.ylabel("Contagem")
plt.title("Distribuição do clearance alvo")
plt.show()



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

sns.scatterplot(ax=axes[0], data=df, x="fu_frac", y="clearance_target", alpha=0.6)
axes[0].set_xlabel("Fração não ligada (fu)")
axes[0].set_ylabel("Clearance (L/h)")
axes[0].set_title("Clearance vs fu")

sns.scatterplot(ax=axes[1], data=df, x="vd_l_kg", y="clearance_target", alpha=0.6, color="#d62728")
axes[1].set_xlabel("Volume de distribuição (L/kg)")
axes[1].set_title("Clearance vs Vd")

plt.tight_layout()
plt.show()



## Observações

- O dataset final possui cobertura integral de SMILES (6.4 mil entradas) e target de clearance disponível para 1.5 mil compostos.
- A distribuição de clearance apresenta cauda longa; recomenda-se normalização logarítmica em pipelines supervisionadas.
- `fu` apresenta correlação negativa moderada com o clearance, enquanto `Vd` exibe ampla variabilidade.
- O arquivo auxiliar `data/processed/pbpk_enriched/pbpk_enriched_v3.npz` armazena embeddings ChemBERTa e alvos, pronto para uso em treinamentos.



## Treinamento DynamicPBPKGNN — Enriched v3

Resultados da rodada batched (batch 24, 200 épocas) usando `data/processed/pbpk_enriched/dynamic_gnn_dataset_enriched_v3.npz`. As curvas e logs abaixo são carregados diretamente de `models/dynamic_gnn_enriched_v3/`.


In [ ]:
import re
from IPython.display import Image

LOG_PATH = BASE_DIR / "models" / "dynamic_gnn_enriched_v3" / "training.log"
TRAINING_CURVE_PATH = BASE_DIR / "models" / "dynamic_gnn_enriched_v3" / "training_curve.png"

pattern_epoch = re.compile(r"Epoch (\d+)/")
pattern_train = re.compile(r"Train Loss: ([0-9.e-]+)")
pattern_val = re.compile(r"Val Loss: ([0-9.e-]+)")

epochs, train_losses, val_losses = [], [], []
current_epoch = None
last_train = None
with LOG_PATH.open() as fh:
    for line in fh:
        if match := pattern_epoch.search(line):
            current_epoch = int(match.group(1))
        elif match := pattern_train.search(line):
            last_train = float(match.group(1))
        elif match := pattern_val.search(line):
            if current_epoch is None or last_train is None:
                continue
            epochs.append(current_epoch)
            train_losses.append(last_train)
            val_losses.append(float(match.group(1)))

loss_df = pd.DataFrame(
    {
        "epoch": epochs,
        "train_loss": train_losses,
        "val_loss": val_losses,
    }
)
loss_df.tail()


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(loss_df["epoch"], loss_df["train_loss"], label="Train Loss", color="#1f77b4")
plt.plot(loss_df["epoch"], loss_df["val_loss"], label="Val Loss", color="#d62728")
plt.xlabel("Epoch")
plt.ylabel("MSE por órgão")
plt.title("DynamicPBPKGNN — Enriched v3 (batch 24)")
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
Image(filename=str(TRAINING_CURVE_PATH))


### Simulações com o checkpoint enriquecido

- CLI validado tanto em GPU quanto em CPU (`logs/dynamic_gnn_enriched_v3_{cuda,cpu}_sim.md`).
- Configuração de teste: dose 100 mg, `CLhepático=12 L/h`, `CLrenal=6 L/h`, 48 passos (Δt = 0,5 h).
- Resultados: `Cmax(blood)=20 mg/L`, `[Final blood]=0,3166 mg/L`, órgãos periféricos com picos ~1,55 mg/L (ossos, pâncreas, pele, tecido adiposo).
- Esses valores servem como baseline para comparar futuras iterações/hyperparameter sweeps.



## Sweep B — hidden_dim=128, layers=4 (em andamento)

A segunda varredura hiperparamétrica expande a dimensionalidade oculta para 128 neurônios e adiciona uma quarta camada de message passing, mantendo `batch_size=24`, `lr=5e-4`, `num_temporal_steps=120` e `dt=0,1`. O objetivo é avaliar ganhos de expressividade mantendo o uso de VRAM próximo de 10 GB. O snapshot abaixo acompanha o log parcial (`models/dynamic_gnn_sweep_b/training.log`) enquanto o treino ainda progride rumo às 200 épocas planificadas.



In [ ]:
SWEEP_B_LOG = BASE_DIR / "models" / "dynamic_gnn_sweep_b" / "training.log"

if SWEEP_B_LOG.exists():
    epochs_b, train_b, val_b = [], [], []
    current_epoch = None
    last_train = None
    with SWEEP_B_LOG.open() as fh:
        for line in fh:
            if match := pattern_epoch.search(line):
                current_epoch = int(match.group(1))
            elif match := pattern_train.search(line):
                last_train = float(match.group(1))
            elif match := pattern_val.search(line):
                if current_epoch is None or last_train is None:
                    continue
                epochs_b.append(current_epoch)
                train_b.append(last_train)
                val_b.append(float(match.group(1)))
    sweep_b_df = pd.DataFrame({"epoch": epochs_b, "train_loss": train_b, "val_loss": val_b})
    display(sweep_b_df.tail())

    if len(sweep_b_df):
        best_idx = sweep_b_df["val_loss"].idxmin()
        best_row = sweep_b_df.loc[best_idx]
        print(
            f"Melhor Val Loss parcial: {best_row.val_loss:.3e} (Epoch {int(best_row.epoch)})"
        )
        print(
            f"Última época registrada: {int(sweep_b_df.iloc[-1].epoch)} | Train={sweep_b_df.iloc[-1].train_loss:.3e} | Val={sweep_b_df.iloc[-1].val_loss:.3e}"
        )
else:
    print("Log do Sweep B ainda não disponível.")



In [ ]:
if SWEEP_B_LOG.exists() and len(sweep_b_df):
    plt.figure(figsize=(8, 4.5))
    plt.plot(sweep_b_df["epoch"], sweep_b_df["train_loss"], label="Train Loss", color="#1f77b4")
    plt.plot(sweep_b_df["epoch"], sweep_b_df["val_loss"], label="Val Loss", color="#ff7f0e")
    plt.xlabel("Epoch")
    plt.ylabel("MSE por órgão")
    plt.title("DynamicPBPKGNN — Sweep B (hidden_dim=128, layers=4)")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Curva ainda indisponível: aguarde o log ser gerado.")



## Sweep C — hidden_dim=160, layers=4 (em andamento)

Varredura paralela com dimensionalidade oculta 160, quatro camadas de message passing, `batch_size=28`, `lr=3e-4`, `num_temporal_steps=120` e `dt=0,1`. Este setup testa um regime mais profundo/robusto mantendo a replicação batched e mira uso de VRAM < 12 GB. Os blocos abaixo lêem `models/dynamic_gnn_sweep_c/training.log` para acompanhar o progresso em tempo quase real.



In [ ]:
SWEEP_C_LOG = BASE_DIR / "models" / "dynamic_gnn_sweep_c" / "training.log"

if SWEEP_C_LOG.exists():
    epochs_c, train_c, val_c = [], [], []
    current_epoch = None
    last_train = None
    with SWEEP_C_LOG.open() as fh:
        for line in fh:
            if match := pattern_epoch.search(line):
                current_epoch = int(match.group(1))
            elif match := pattern_train.search(line):
                last_train = float(match.group(1))
            elif match := pattern_val.search(line):
                if current_epoch is None or last_train is None:
                    continue
                epochs_c.append(current_epoch)
                train_c.append(last_train)
                val_c.append(float(match.group(1)))
    sweep_c_df = pd.DataFrame({"epoch": epochs_c, "train_loss": train_c, "val_loss": val_c})
    display(sweep_c_df.tail())

    if len(sweep_c_df):
        best_idx = sweep_c_df["val_loss"].idxmin()
        best_row = sweep_c_df.loc[best_idx]
        print(
            f"Melhor Val Loss parcial: {best_row.val_loss:.3e} (Epoch {int(best_row.epoch)})"
        )
        print(
            f"Última época registrada: {int(sweep_c_df.iloc[-1].epoch)} | Train={sweep_c_df.iloc[-1].train_loss:.3e} | Val={sweep_c_df.iloc[-1].val_loss:.3e}"
        )
else:
    print("Log do Sweep C ainda não disponível.")



In [ ]:
if SWEEP_C_LOG.exists() and len(sweep_c_df):
    plt.figure(figsize=(8, 4.5))
    plt.plot(sweep_c_df["epoch"], sweep_c_df["train_loss"], label="Train Loss", color="#2ca02c")
    plt.plot(sweep_c_df["epoch"], sweep_c_df["val_loss"], label="Val Loss", color="#9467bd")
    plt.xlabel("Epoch")
    plt.ylabel("MSE por órgão")
    plt.title("DynamicPBPKGNN — Sweep C (hidden_dim=160, layers=4)")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Curva ainda indisponível: aguarde o log ser gerado.")



## Histórico de treinos e correções metodológicas (DynamicPBPKGNN)

- **v3 (original)**: dataset `dynamic_gnn_dataset_enriched_v3.npz` gerado a partir de `pbpk_parameters_wide_enriched_v3.csv` com dose fixa (100 mg), Kp homogêneos escalonados por Vd e split aleatório → alta redundância de parâmetros (257 combinações únicas em 6.551 amostras) e vazamento entre treino/val (mesmos parâmetros em ambos os splits), resultando em R² ≈ 1,0.
- **v3_dedup**: versão deduplicada (`dynamic_gnn_dataset_enriched_v3_dedup.npz`) removendo clones de parâmetros; treino com split por grupos de parâmetros (hash) e validação robusta ainda mostra R² muito próximo de 1,0, mas já sem leak explícito por hash.
- **v4_compound (atual)**: novo dataset `dynamic_gnn_dataset_enriched_v4.npz` com dose variável (50–200 mg), ruído fisiológico em Kp/clearances e `compound_ids` explícitos. O treino em `models/dynamic_gnn_v4_compound/` usa `--split-strategy compound`, impondo separação estrita por composto entre treino e validação. As métricas deste setup serão a referência “científica” para comparação com v3/v3_dedup e para qualquer relato de desempenho em manuscritos.



## Avaliação robusta dos modelos (sweep_b e sweep_c)

Avaliação realizada com:
- Split por grupos de parâmetros (evita leak)
- Métricas por janelas temporais (1-12h, 12-24h, 24-48h, 48-100h)
- Escalas linear e log1p
- Comparação com baseline mean (média do treino por órgão×tempo)



In [ ]:
import pandas as pd
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Carregar tabela comparativa
comp_df = pd.read_csv('models/comparison_robust/comparison_table.csv')
print("📊 Tabela comparativa de métricas:")
print(comp_df.to_string(index=False))

# Carregar resumo
with open('models/comparison_robust/comparison_summary.json', 'r') as f:
    summary = json.load(f)

print("\n📈 Resumo por modelo:")
for model in summary['r2_linear_mean'].keys():
    print(f"\n{model}:")
    print(f"  R² (linear) médio: {summary['r2_linear_mean'][model]:.6f}")
    print(f"  R² (log1p) médio: {summary['r2_log1p_mean'][model]:.6f}")
    print(f"  MSE médio: {summary['mse_mean'][model]:.2e}")



In [ ]:
# Visualizar gráficos comparativos
from IPython.display import Image, display

img_path = Path('models/comparison_robust/comparison_plots.png')
if img_path.exists():
    display(Image(str(img_path)))
else:
    print("Gráfico ainda não disponível.")



### Observações sobre os resultados

**Sweep B e Sweep C (dataset v3, split por grupos):**
- R² ainda muito alto (~0.99999) mesmo com split por grupos
- Baseline mean tem R² ~0.88-0.99, indicando que o problema é inerentemente "fácil" para o modelo
- MSE extremamente baixo (~10⁻⁷), sugerindo que o modelo está quase perfeito no dataset v3

**Aguardando v4_compound:**
- O dataset v4 (dose variável, ruído fisiológico, split por composto) deve fornecer métricas mais realistas
- A avaliação robusta do v4_compound será a referência científica final



## Resultados finais: v4_compound avaliado

O treino v4_compound foi concluído e avaliado robustamente. Comparação completa incluindo os três modelos:



In [ ]:
# Carregar comparação completa (incluindo v4_compound)
comp_all_df = pd.read_csv('models/comparison_robust_all/comparison_table.csv')
print("📊 Tabela comparativa completa (sweep_b, sweep_c, v4_compound):")
print(comp_all_df.to_string(index=False))

# Carregar resumo completo
with open('models/comparison_robust_all/comparison_summary.json', 'r') as f:
    summary_all = json.load(f)

print("\n📈 Resumo por modelo (médias):")
for model in summary_all['r2_linear_mean'].keys():
    print(f"\n{model}:")
    print(f"  R² (linear) médio: {summary_all['r2_linear_mean'][model]:.6f}")
    print(f"  R² (log1p) médio: {summary_all['r2_log1p_mean'][model]:.6f}")
    print(f"  MSE médio: {summary_all['mse_mean'][model]:.2e}")



In [ ]:
# Visualizar gráficos comparativos completos
img_path_all = Path('models/comparison_robust_all/comparison_plots.png')
if img_path_all.exists():
    display(Image(str(img_path_all)))
else:
    print("Gráfico ainda não disponível.")



### Análise dos resultados finais

**v4_compound (dataset v4, split por composto):**
- R² médio: ~0.999993 (ainda muito alto, mas ligeiramente melhor que sweep_b/c)
- Baseline mean R² na primeira janela: **0.944** (vs 0.878 em v3), indicando que o dataset v4 é mais desafiador
- Split: 6,551 compostos únicos, 5,241 train / 1,310 val (separação estrita por composto)
- MSE médio: ~4.07×10⁻⁷ (ligeiramente maior que v3, mas ainda muito baixo)

**Observações críticas:**
1. **R² ainda muito alto**: Mesmo com split por composto, dose variável e ruído fisiológico, o modelo alcança R² ~0.99999
2. **Baseline mean melhor**: O baseline mean no v4 tem R² ~0.94-0.99, indicando que o problema é inerentemente "fácil" para qualquer modelo
3. **Possíveis causas**:
   - Dataset gerado por simulação determinística (distillation) pode ser muito regular
   - Ruído adicionado pode ser insuficiente para criar desafio real
   - Modelo pode estar aprendendo padrões muito simples (ex: clearance → concentração)
4. **Próximos passos científicos**:
   - Avaliar em dados experimentais reais (não simulados)
   - Adicionar mais diversidade/ruído ao dataset
   - Comparar com modelos mais simples (regressão linear, kNN) para verificar se o problema é trivial
   - Implementar métricas mais rigorosas (ex: R² em escala log, threshold de concentração)



## Avaliação Científica Rigorosa (Métricas Regulatórias)

Avaliação realizada com métricas científicas adequadas para PBPK (Fold Error, GMFE), conforme padrões FDA/EMA.



In [ ]:
# Carregar resultados da avaliação científica
import json
from pathlib import Path

scientific_eval_path = Path('models/dynamic_gnn_v4_compound/evaluation_scientific/scientific_eval.json')
if scientific_eval_path.exists():
    with open(scientific_eval_path, 'r') as f:
        scientific_results = json.load(f)

    print("📊 AVALIAÇÃO CIENTÍFICA - Métricas Regulatórias (FDA/EMA)")
    print("=" * 70)
    print("\n🎯 Modelo DynamicPBPKGNN v4_compound:")
    model_metrics = scientific_results['model_metrics']
    print(f"  Fold Error (FE) médio: {model_metrics['fold_error_mean']:.4f}")
    print(f"  Fold Error (FE) mediano: {model_metrics['fold_error_median']:.4f}")
    print(f"  Fold Error (FE) p67: {model_metrics['fold_error_p67']:.4f}")
    print(f"  Geometric Mean Fold Error (GMFE): {model_metrics['geometric_mean_fold_error']:.4f}")
    print(f"  % dentro de 1.25×: {model_metrics['percent_within_1.25x']:.2f}%")
    print(f"  % dentro de 1.5×: {model_metrics['percent_within_1.5x']:.2f}%")
    print(f"  % dentro de 2.0×: {model_metrics['percent_within_2.0x']:.2f}%")
    print(f"  R²: {model_metrics['r2']:.6f}")
    print(f"  MAE: {model_metrics['mae']:.6f}")
    print(f"  RMSE: {model_metrics['rmse']:.6f}")
    print(f"  Número de previsões: {model_metrics['num_predictions']:,}")

    print("\n📊 Baseline (Regressão Linear):")
    baseline_metrics = scientific_results['baseline_linear_metrics']
    print(f"  Fold Error (FE) médio: {baseline_metrics['fold_error_mean']:.4f}")
    print(f"  Geometric Mean Fold Error (GMFE): {baseline_metrics['geometric_mean_fold_error']:.4f}")
    print(f"  % dentro de 2.0×: {baseline_metrics['percent_within_2.0x']:.2f}%")
    print(f"  R²: {baseline_metrics['r2']:.6f}")

    print("\n✅ Critérios de Aceitação (FDA/EMA):")
    print(f"  FE ≤ 2.0: {'✅ PASSOU' if model_metrics['fold_error_p67'] <= 2.0 else '❌ FALHOU'}")
    print(f"  GMFE < 2.0: {'✅ PASSOU' if model_metrics['geometric_mean_fold_error'] < 2.0 else '❌ FALHOU'}")
    print(f"  % dentro de 2.0× ≥ 67%: {'✅ PASSOU' if model_metrics['percent_within_2.0x'] >= 67.0 else '❌ FALHOU'}")
else:
    print("⚠️  Avaliação científica ainda não disponível")



In [ ]:
# Visualizar gráficos científicos
scientific_plots = [
    'models/dynamic_gnn_v4_compound/evaluation_scientific/scatter_pred_vs_obs.png',
    'models/dynamic_gnn_v4_compound/evaluation_scientific/fold_error_distribution.png',
    'models/dynamic_gnn_v4_compound/evaluation_scientific/residuals_vs_predicted.png'
]

for plot_path in scientific_plots:
    if Path(plot_path).exists():
        print(f"\n📈 {Path(plot_path).name}:")
        display(Image(plot_path))
    else:
        print(f"⚠️  {plot_path} não encontrado")

